In [ ]:
# training.ipynb
# Stage 2 of 2: the full sweep via the supervisor. Run AFTER
# prepare_training.ipynb (which builds the H5 AND smoke-validates the training
# pipeline), so this notebook is just the multi-hour run + resume.
REPO = "/workspace/stable-query-latent"
URL = "https://github.com/Nice9Tian/stable-query-latent.git"
LOG = "/workspace/stable_query_latent_logs/pipeline.log"
print('repo:', REPO)
print('log :', LOG)

In [ ]:
# Clone or pull before every run.
import os

if not os.path.isdir(os.path.join(REPO, ".git")):
    !git clone {URL} {REPO}

%cd {REPO}
!git remote set-url origin {URL}
!git remote -v
!git pull origin main
!git status --short
!git rev-parse HEAD


In [ ]:
# Start GPU + CPU + RAM + disk I/O monitor.
import subprocess, threading, time, psutil
from pathlib import Path

stop = False

def _read_int(path):
    try:
        text = Path(path).read_text().strip()
        if text == "max":
            return None
        return int(text)
    except Exception:
        return None

def get_memory_status():
    limit = _read_int("/sys/fs/cgroup/memory.max")
    used = _read_int("/sys/fs/cgroup/memory.current")
    if limit is None or used is None:
        limit = _read_int("/sys/fs/cgroup/memory/memory.limit_in_bytes")
        used = _read_int("/sys/fs/cgroup/memory/memory.usage_in_bytes")
    if limit and used and limit < 10**18:
        return used / limit * 100, used / 1024**3, limit / 1024**3, "cgroup"
    vm = psutil.virtual_memory()
    return vm.percent, vm.used / 1024**3, vm.total / 1024**3, "host"

def get_gpu_status():
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total", "--format=csv,noheader,nounits"],
            capture_output=True,
            text=True,
            timeout=3,
        ).stdout.strip()
        parts = [p.strip() for p in out.splitlines()[0].split(",")]
        return f"{float(parts[0]):.0f}%, {float(parts[1])/1024:.1f}/{float(parts[2])/1024:.1f} GiB"
    except Exception as e:
        return f"n/a ({e})"

def monitor(interval=5):
    last_disk = psutil.disk_io_counters()
    last_t = time.time()
    psutil.cpu_percent(interval=None)
    while not stop:
        gpu = get_gpu_status()
        cpu = psutil.cpu_percent(interval=None)
        ram_pct, ram_used, ram_total, ram_source = get_memory_status()
        now_disk = psutil.disk_io_counters()
        now_t = time.time()
        dt = max(now_t - last_t, 1e-6)
        read_mb = (now_disk.read_bytes - last_disk.read_bytes) / 1e6 / dt
        write_mb = (now_disk.write_bytes - last_disk.write_bytes) / 1e6 / dt
        last_disk, last_t = now_disk, now_t
        print(
            f"[gpu] {gpu} | [cpu] {cpu:.0f}% | "
            f"[ram:{ram_source}] {ram_pct:.0f}% ({ram_used:.1f}/{ram_total:.1f} GiB) | "
            f"[disk] R {read_mb:.1f} MB/s W {write_mb:.1f} MB/s",
            flush=True,
        )
        time.sleep(interval)

threading.Thread(target=monitor, daemon=True).start()


In [ ]:
# Stage the embedding H5 to FAST LOCAL disk (overlay/NVMe) if it fits; else fall
# back to the network volume (/workspace = MooseFS, ~256MB/s read cap). Only the
# big READ (training) is redirected via --h5; checkpoints/ledger keep writing to
# out_dir on /workspace. Idempotent: skips if prepare_training already staged it
# this session. Non-fatal: any failure -> H5_PATH stays the /workspace H5.
import os, threading, time

WORKSPACE_H5 = os.path.join(REPO, "game_review_data/embedding_h5.h5")
LOCAL_H5 = "/root/data/embedding_h5.h5"
H5_PATH = WORKSPACE_H5   # safe default / fallback


def _free_bytes(path):
    st = os.statvfs(path)
    return st.f_bavail * st.f_frsize


def parallel_copy(src, dst, workers=8, chunk=64 << 20):
    """Thread-parallel byte-range copy (file I/O releases the GIL, so threads
    overlap the network reads; no fork/CUDA hazard from the notebook kernel)."""
    size = os.path.getsize(src)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    with open(dst, "wb") as f:
        f.truncate(size)                       # preallocate
    bounds = [(i * size // workers, (i + 1) * size // workers) for i in range(workers)]
    errors = []

    def copy_range(start, end):
        sfd = os.open(src, os.O_RDONLY)
        dfd = os.open(dst, os.O_WRONLY)
        try:
            off = start
            while off < end:
                data = os.pread(sfd, min(chunk, end - off), off)
                if not data:
                    break
                os.pwrite(dfd, data, off)
                off += len(data)
        except BaseException as exc:            # noqa: BLE001
            errors.append(exc)
        finally:
            os.close(sfd)
            os.close(dfd)

    threads = [threading.Thread(target=copy_range, args=b) for b in bounds]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    if errors:
        raise errors[0]
    if os.path.getsize(dst) != size:
        raise IOError("size mismatch after copy")


try:
    src_size = os.path.getsize(WORKSPACE_H5)
    if os.path.exists(LOCAL_H5) and os.path.getsize(LOCAL_H5) == src_size:
        H5_PATH = LOCAL_H5
        print(f"local H5 already staged: {LOCAL_H5} ({src_size / 2**30:.1f} GiB)")
    else:
        free = _free_bytes("/")
        need = src_size + (5 << 30)            # 5 GiB headroom
        print(f"H5={src_size / 2**30:.1f}GiB  local('/') free={free / 2**30:.1f}GiB  need~{need / 2**30:.1f}GiB")
        if free >= need:
            t0 = time.time()
            parallel_copy(WORKSPACE_H5, LOCAL_H5, workers=8)
            dt = max(time.time() - t0, 1e-6)
            H5_PATH = LOCAL_H5
            print(f"staged -> {LOCAL_H5} in {dt:.0f}s ({src_size / 2**30 / dt:.2f} GiB/s)")
        else:
            print("not enough local space -> using /workspace H5 (expand Container Disk to stage locally)")
except BaseException as exc:
    print(f"staging failed ({type(exc).__name__}: {exc}) -> using /workspace H5")
    try:
        if os.path.exists(LOCAL_H5) and os.path.getsize(LOCAL_H5) != os.path.getsize(WORKSPACE_H5):
            os.remove(LOCAL_H5)
    except OSError:
        pass
    H5_PATH = WORKSPACE_H5

print("H5_PATH =", H5_PATH)


In [ ]:
# (Optional) Inspect the memory plan before the full run. Calibrates (C,R) on
# pseudo batches, then prints per combo the chosen backward-mode + stem chunk
# size. Safe to skip -- the supervisor calibrates + plans automatically.
# Uses the staged local H5 (H5_PATH).
!python -u VICReg_review/oom_proxy.py \
  --h5 {H5_PATH} \
  --calib-out VICReg_review/heads/cloud_full_sweep_a100/calib.json --measure
!python -u -m VICReg_review.sweep.planner \
  --config VICReg_review/sweep/sweep.yaml \
  --h5 {H5_PATH} \
  --calib-json VICReg_review/heads/cloud_full_sweep_a100/calib.json


In [ ]:
# Train the full sweep via the NEW supervisor (clean redesign; replaces sweep_cloud).
# Reads VICReg_review/sweep/sweep.yaml; --h5 redirects the big READ to the staged
# local copy (H5_PATH) while checkpoints/ledger keep writing to out_dir on /workspace.
# Spawns a persistent worker that calibrates once, then trains every combo with
# the chunked stem -> FULL data, NO sentence cap. Crash-isolated: a worker
# OOM/crash makes the supervisor shrink the stem chunk (or RAM-downgrade on
# SIGKILL), restart the worker, and resume from the ledger; one bad combo is
# marked 'failed' and never aborts the run.
# Resume: just re-run this cell -- combos verified done (vs checkpoint) are skipped.
# Probes are emitted to the queue and drained later in eval.ipynb.
# State: VICReg_review/heads/cloud_full_sweep_a100/{ledger.jsonl, calib.json, sweep_jobs/}
!python -u VICReg_review/sweep/supervisor.py \
  --config VICReg_review/sweep/sweep.yaml \
  --h5 {H5_PATH} \
  --logout-address {LOG}


In [ ]:
# Training done. Stop the monitor and print the ledger summary.
# Probe draining + final eval + artifact collection live in eval.ipynb
# (run it next, on this same pod, after training finishes).
stop = True

import json
from collections import Counter
from pathlib import Path

out_dir = Path(REPO, 'VICReg_review/heads/cloud_full_sweep_a100')
ledger = out_dir / 'ledger.jsonl'
queue_dir = out_dir / 'probe_queue'

if ledger.exists():
    latest = {}
    for line in ledger.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            rec = json.loads(line)
        except json.JSONDecodeError:
            continue
        if rec.get('combo_id'):
            latest[rec['combo_id']] = rec
    counts = Counter(r.get('status', '?') for r in latest.values())
    print(f'ledger: {dict(counts)} ({len(latest)} combos)')
    failed = [c for c, r in latest.items() if r.get('status') == 'failed']
    if failed:
        print(f'failed combos ({len(failed)}):', failed)
else:
    print('no ledger yet:', ledger)

pending = sorted(queue_dir.glob('*.json')) if queue_dir.exists() else []
print(f'probe jobs queued (drain these in eval.ipynb): {len(pending)}')
print('Next: run Pod/eval.ipynb on this pod to drain probes, run final eval, and archive.')
